# Tome C1, Chapter 12: Designs and Finite Geometries (Python companion)

Exercises first, solutions at the end. We build Steiner triple systems by the constructions of Bose and Skolem, solve Kirkman's schoolgirl problem, construct projective planes over finite fields, turn them into orthogonal Latin squares and back, prove Tarry's theorem on the thirty-six officers by exhaustive search, tabulate the Bruck–Ryser theorem, and generate test suites for pairwise testing.

*Before each solution, predict what the output will look like.*

In [ ]:
import itertools, random, math, time
from collections import Counter
import numpy as np
import matplotlib.pyplot as plt
random.seed(12)

## Exercises

**P12.1 (★) Triple systems.** Implement the constructions of Theorems 12.5 (Bose) and 12.7 (Skolem) and check that they give an $\mathrm{STS}(v)$ for every $v\equiv1,3\pmod6$ up to $69$.

**P12.2 (★★) Kirkman's schoolgirls.** The lines of the projective space $\mathrm{PG}(3,2)$ (triples $\{a,b,a\oplus b\}$ of non-zero vectors of $\mathbb F_2^4$) form an $\mathrm{STS}(15)$. Find $7$ *spreads* (sets of $5$ disjoint lines) using every line once, and check Table 12.1.

**P12.3 (★) Projective planes.** Build $\mathrm{PG}(2,q)$ for $q=2,3,4,5,7$ (with a hand-made $\mathbb F_4$), check the axioms and the counts of Proposition 12.9, and find a set of $q+2$ points with no three collinear in $\mathrm{PG}(2,4)$ (Exercise 12.7).

**P12.4 (★★) From planes to squares and back.** Build $q-1$ MOLS of order $q$ (Corollary 12.16), turn them into an affine plane and check the axioms (Theorem 12.15). Build $2$ MOLS of order $12$ by MacNeish's product.

**P12.5 (★★★) The thirty-six officers.** Prove by exhaustive search that no Latin square of order $6$ has an orthogonal mate (Tarry, 1900). A square has a mate exactly when its cells split into $n$ disjoint *transversals* (one cell per row and column, all symbols different), and it is enough to examine reduced squares.

**P12.6 (★) Bruck–Ryser.** List the orders $n\le100$ excluded by Theorem 12.19, and the non-prime-power orders that remain undecided.

**P12.7 (★★) Pairwise testing.** Compare, for binary and ternary parameters, the sizes of covering arrays from MOLS, from Exercise 12.9, from the random bound of Theorem 12.23, and from the greedy algorithm of Exercise 12.10.

## Solutions

### P12.1 Triple systems

In [ ]:
def bose(m):
    n = 2 * m + 1; op = lambda x, y: (m + 1) * (x + y) % n
    B = [frozenset({(x, 0), (x, 1), (x, 2)}) for x in range(n)]
    B += [frozenset({(x, i), (y, i), (op(x, y), (i + 1) % 3)}) for x, y in itertools.combinations(range(n), 2) for i in range(3)]
    return [(x, i) for x in range(n) for i in range(3)], B
def skolem(m):
    n = 2 * m; sig = lambda s: s // 2 if s % 2 == 0 else m + s // 2
    op = lambda x, y: sig((x + y) % n)
    B = [frozenset({(x, 0), (x, 1), (x, 2)}) for x in range(m)]
    B += [frozenset({'inf', (m + x, i), (x, (i + 1) % 3)}) for x in range(m) for i in range(3)]
    B += [frozenset({(x, i), (y, i), (op(x, y), (i + 1) % 3)}) for x, y in itertools.combinations(range(n), 2) for i in range(3)]
    return ['inf'] + [(x, i) for x in range(n) for i in range(3)], B
def is_sts(P, B):
    cnt = Counter(frozenset(p) for b in B for p in itertools.combinations(b, 2))
    return all(len(b) == 3 for b in B) and len(cnt) == len(P) * (len(P) - 1) // 2 and set(cnt.values()) == {1}
for v in range(3, 70):
    if v % 6 == 3:
        P, B = bose((v - 3) // 6)
    elif v % 6 == 1 and v > 1:
        P, B = skolem((v - 1) // 6)
    else:
        continue
    assert len(P) == v and len(B) == v * (v - 1) // 6 and is_sts(P, B), v
print('Bose and Skolem give an STS(v) for every v = 1, 3 mod 6 from 3 to 69')

### P12.2 Kirkman's schoolgirls

The search is an exact cover in two stages: first all $56$ spreads of $\mathrm{PG}(3,2)$, then $7$ of them partitioning the $35$ lines.

In [ ]:
pts = list(range(1, 16))
lines = sorted({frozenset((a, b, a ^ b)) for a in pts for b in pts if a < b}, key=sorted)
spreads = []
def find_spreads(cur, used):
    if len(used) == 15:
        spreads.append(frozenset(cur)); return
    p = min(set(pts) - used)
    for L in lines:
        if p in L and not (L & used):
            find_spreads(cur + [L], used | L)
find_spreads([], frozenset())
def packing(cur, used):
    if len(cur) == 7:
        return cur
    L0 = min((L for L in lines if L not in used), key=sorted)
    for S in spreads:
        if L0 in S and not (S & used):
            r = packing(cur + [S], used | S)
            if r:
                return r
days = packing([], frozenset())
names = 'ABCDEFGHIJKLMNO'
table = [' '.join(''.join(sorted(names[p - 1] for p in L)) for L in sorted(S, key=min)) for S in days]
print(len(lines), 'lines,', len(spreads), 'spreads')
for d, row in zip(['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun'], table):
    print(d, row)
book = ['ABC DHL EJO FKM GIN', 'ADE BHJ CMN FIO GKL', 'AFG BIK CLO DJN EHM', 'AHI BLN CEF DKO GJM',
        'AJK BMO CDG EIL FHN', 'ALM BDF CIJ EKN GHO', 'ANO BEG CHK DIM FJL']
pairs = Counter(frozenset(p) for row in book for t in row.split() for p in itertools.combinations(t, 2))
print('Table 12.1 equals this solution:', table == book, '| every pair once:', len(pairs) == 105 and set(pairs.values()) == {1},
      '| every girl once a day:', all(sorted(''.join(r.split())) == list(names) for r in book))

### P12.3 Projective planes

In [ ]:
class Field:
    # F_p for p prime, or F_4 = {0,1,w,w^2} coded 0,1,2,3 with addition = xor
    def __init__(self, q):
        self.q = q
        if q == 4:
            exp = [1, 2, 3]; log = {1: 0, 2: 1, 3: 2}
            self.add = lambda a, b: a ^ b
            self.mul = lambda a, b: 0 if 0 in (a, b) else exp[(log[a] + log[b]) % 3]
        else:
            self.add = lambda a, b: (a + b) % q
            self.mul = lambda a, b: a * b % q
    def dot(self, u, v):
        s = 0
        for a, b in zip(u, v):
            s = self.add(s, self.mul(a, b))
        return s
def normalised(F):
    # one representative per 1-dim subspace: first non-zero coordinate equal to 1
    return [v for v in itertools.product(range(F.q), repeat=3) if any(v) and v[next(i for i in range(3) if v[i])] == 1]
def PG(q):
    F = Field(q); P = normalised(F)
    lines = [frozenset(i for i, x in enumerate(P) if F.dot(a, x) == 0) for a in P]
    return F, P, lines
for q in [2, 3, 4, 5, 7]:
    F, P, Ls = PG(q)
    ok1 = all(sum(1 for L in Ls if i in L and j in L) == 1 for i, j in itertools.combinations(range(len(P)), 2))
    ok2 = all(len(L & M) == 1 for L, M in itertools.combinations(Ls, 2))
    print(f'PG(2,{q}): {len(P)} points, {len(Ls)} lines, line sizes {set(map(len, Ls))}, '
          f'points per line pair 1: {ok2}, lines per point pair 1: {ok1}')

In [ ]:
F, P, Ls = PG(4)
def arc(S):
    return all(len(L & S) <= 2 for L in Ls)
conic = frozenset(i for i, (x, y, z) in enumerate(P) if F.add(F.mul(y, y), F.mul(x, z)) == 0)   # y^2 = xz
print('conic in PG(2,4):', len(conic), 'points, no three collinear:', arc(conic))
nucleus = [i for i in range(len(P)) if i not in conic and arc(conic | {i})]
print('points that extend it to a 6-point hyperoval:', [P[i] for i in nucleus])

In characteristic $2$ all tangent lines to a conic pass through one point, the *nucleus*, and adding it gives $q+2$ points with no three collinear: the bound of Exercise 12.7 is reached for even $q$.

### P12.4 From planes to squares and back

In [ ]:
def field_mols(q):
    F = Field(q)
    return [[[F.add(F.mul(a, x), y) for y in range(q)] for x in range(q)] for a in range(1, q)]
def is_latin(L):
    n = len(L)
    return all(len(set(r)) == n for r in L) and all(len({L[i][j] for i in range(n)}) == n for j in range(n))
def orthogonal(L, M):
    n = len(L)
    return len({(L[i][j], M[i][j]) for i in range(n) for j in range(n)}) == n * n
def plane_from_mols(Ms):
    n = len(Ms[0]); pts = [(i, j) for i in range(n) for j in range(n)]
    lines = [frozenset((i, j) for j in range(n)) for i in range(n)] + [frozenset((i, j) for i in range(n)) for j in range(n)]
    lines += [frozenset(p for p in pts if M[p[0]][p[1]] == s) for M in Ms for s in range(n)]
    return pts, lines
def is_affine_plane(pts, lines):
    n2 = len(pts)
    a1 = all(sum(1 for L in lines if p in L and r in L) == 1 for p, r in itertools.combinations(pts, 2))
    a2 = all(sum(1 for M in lines if p in M and not (M & L)) == 1 for L in lines for p in pts if p not in L)
    return a1 and a2
for q in [2, 3, 4, 5, 7]:
    Ms = field_mols(q)
    ok = all(map(is_latin, Ms)) and all(orthogonal(A, B) for A, B in itertools.combinations(Ms, 2))
    pts, lines = plane_from_mols(Ms)
    print(f'q={q}: {len(Ms)} MOLS: {ok};  affine plane with {len(pts)} points and {len(lines)} lines: {is_affine_plane(pts, lines)}')

In [ ]:
def kron(L, M):
    m, n = len(L), len(M)
    return [[L[i // n][j // n] * n + M[i % n][j % n] for j in range(m * n)] for i in range(m * n)]
A4, B4, _ = field_mols(4); A3, B3 = field_mols(3)
L1, L2 = kron(A4, A3), kron(B4, B3)
print('order 12 from MacNeish: Latin', is_latin(L1), is_latin(L2), ' orthogonal', orthogonal(L1, L2))

### P12.5 The thirty-six officers

A Latin square $L$ has an orthogonal mate $M$ exactly when its cells split into $n$ disjoint transversals: the cells where $M$ takes a given symbol form a transversal of $L$. Permuting rows, columns and symbols preserves this, so it suffices to check the *reduced* squares, with first row and first column $0,1,\dots,n-1$.

In [ ]:
def reduced_squares(n):
    L = [[None] * n for _ in range(n)]
    for j in range(n):
        L[0][j] = j
    for i in range(n):
        L[i][0] = i
    cells = [(i, j) for i in range(1, n) for j in range(1, n)]
    out = []
    def rec(k):
        if k == len(cells):
            out.append([r[:] for r in L]); return
        i, j = cells[k]
        used = {L[i][c] for c in range(j)} | {L[r][j] for r in range(i)}
        for s in range(n):
            if s not in used:
                L[i][j] = s; rec(k + 1)
        L[i][j] = None
    rec(0)
    return out
def transversals(L):
    n = len(L)
    return [p for p in itertools.permutations(range(n)) if len({L[i][p[i]] for i in range(n)}) == n]
def has_mate(L):
    n = len(L); T = [frozenset((i, p[i]) for i in range(n)) for p in transversals(L)]
    def rec(used, k, start):
        if k == n:
            return True
        cell = min(c for c in itertools.product(range(n), repeat=2) if c not in used)
        return any(rec(used | t, k + 1, 0) for t in T if cell in t and not (t & used))
    return rec(frozenset(), 0, 0)
for n in [3, 4, 5]:
    R = reduced_squares(n)
    print(f'order {n}: {len(R)} reduced squares, {sum(map(has_mate, R))} with an orthogonal mate')
t0 = time.time(); R6 = reduced_squares(6)
mates = sum(map(has_mate, R6))
print(f'order 6: {len(R6)} reduced squares, {mates} with an orthogonal mate  ({time.time()-t0:.0f} s)')

None of the $9408$ reduced Latin squares of order $6$ has a mate: Euler's officers cannot be arranged. Tarry settled the question by hand in 1900, sorting the squares into classes; Stinson (1984) gave a short proof without computers. Euler already knew a simpler fact: the cyclic square $L(i,j)=i+j \bmod n$ of even order has no transversal at all, since the symbols of a transversal would add up to $\sum_i i+\sum_j j\equiv0$, while $0+1+\dots+(n-1)\equiv\frac n2\pmod n$.

### P12.6 Bruck–Ryser

In [ ]:
def is_prime_power(n):
    for p in range(2, n + 1):
        if n % p == 0:
            while n % p == 0:
                n //= p
            return n == 1
    return False
def sum_two_squares(n):
    return any(int(math.isqrt(n - a * a)) ** 2 == n - a * a for a in range(int(math.isqrt(n)) + 1))
excluded = [n for n in range(2, 101) if n % 4 in (1, 2) and not sum_two_squares(n)]
open_ = [n for n in range(2, 101) if not is_prime_power(n) and n not in excluded and n != 10]
print('excluded by Bruck-Ryser:', excluded)
print('not prime powers, not excluded (10 is ruled out by Lam):', open_)

### P12.7 Pairwise testing

In [ ]:
def is_ca(A, v):
    k = len(A[0])
    return all(len({(r[c], r[d]) for r in A}) == v * v for c, d in itertools.combinations(range(k), 2))
def greedy_ca(k, v):
    # Exercise 12.10: rows chosen entry by entry by conditional expectation
    U = {(c, d, s, t) for c, d in itertools.combinations(range(k), 2) for s in range(v) for t in range(v)}
    rows = []
    while U:
        row = [None] * k
        for c in range(k):
            best, bv = None, -1
            for s in range(v):
                row[c] = s; e = 0.0
                for (a, b, x, y) in U:
                    pa = 1.0 if row[a] == x else (0.0 if row[a] is not None else 1 / v)
                    pb = 1.0 if row[b] == y else (0.0 if row[b] is not None else 1 / v)
                    e += pa * pb
                if e > bv:
                    best, bv = s, e
            row[c] = best
        rows.append(row)
        U = {(a, b, x, y) for (a, b, x, y) in U if not (row[a] == x and row[b] == y)}
    return rows
def binary_ca(N):
    s = (N + 1) // 2
    cols = [[0] + [1 if r in S else 0 for r in range(1, N)] for S in itertools.combinations(range(1, N), s)]
    return [list(r) for r in zip(*cols)]
def min_binary(k):
    N = 3
    while math.comb(N - 1, (N + 1) // 2) < k:
        N += 1
    return N
print(' k | binary: exact  greedy  random bound | ternary: greedy  random bound  (MOLS: 9 rows for k<=4)')
for k in [4, 10, 20, 40, 80]:
    g2, g3 = greedy_ca(k, 2), greedy_ca(k, 3)
    assert is_ca(g2, 2) and is_ca(g3, 3)
    b2 = math.ceil(4 * math.log(math.comb(k, 2) * 4)); b3 = math.ceil(9 * math.log(math.comb(k, 2) * 9))
    print(f'{k:3d} |        {min_binary(k):4d}  {len(g2):6d}  {b2:12d} |          {len(g3):6d}  {b3:12d}')
B = binary_ca(14)
print('Exercise 12.9 with N = 14:', len(B[0]), 'binary columns, covering array:', is_ca(B, 2))

The greedy suites are much smaller than the random bound (which is only an upper bound), and for binary parameters they stay within a few rows of the exact Katona–Kleitman–Spencer answer. Orthogonal Latin squares win when they apply: $9$ tests for $4$ ternary parameters, every pair of values of every pair of parameters exactly once.